# 🧠 ARCHE3-7B — FCT Training + AGI Score Benchmark
**Two-stage notebook:**
1. **Hive FCT Training** — target-only loss on UPDATE / ACT / RIPPLE / ANALOGY tokens only
2. **AGI Score Test** — scale from 0 (calculator) to 100 (AGI)

| Range | Level |
|---|---|
| 0–10 | Calculator / lookup table |
| 11–25 | Statistical parrot |
| 26–40 | Language model (GPT-2 level) |
| 41–55 | Strong LLM (GPT-4 level) |
| 56–70 | Proto-intellect |
| 71–85 | AGI seeds |
| 86–100 | AGI |

**Storage:** everything saved to `/content/arche3` (local Colab disk, ~104 GB).  
Checkpoints and results are downloaded at the end — no Google Drive needed.

## 1. Install dependencies

In [ ]:
!pip install torch numpy psutil matplotlib seaborn pandas --quiet

import torch, os, sys

WORK_DIR = '/content/arche3'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

import shutil
total, used, free = shutil.disk_usage('/content')
print(f'Disk    : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

## 2. Upload project files

In [ ]:
from google.colab import files

REQUIRED = [
    'arche3_config.py',
    'arche3_model.py',
    'hive_trainer.py',
    'hive_router.py',
    'arche3_tokenizer.py',
    'dense_core_store.py',
    'hive_store.py',
    'data_loader.py',
]

print('Upload ALL of these files:')
for f in REQUIRED:
    print(f'  {f}')
print('  worldmodel_core_1000.jsonl  (upload separately in next cell)')

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(WORK_DIR, name)
    with open(path, 'wb') as f:
        f.write(data)

missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    print(f'\n⚠  Missing: {missing}')
else:
    print('✅ All source files present')

In [ ]:
os.makedirs('data_corpus', exist_ok=True)
JSONL_PATH = 'data_corpus/worldmodel_core_1000.jsonl'

if not os.path.exists(JSONL_PATH):
    print('Upload worldmodel_core_1000.jsonl:')
    up = files.upload()
    for name, data in up.items():
        with open(JSONL_PATH, 'wb') as f:
            f.write(data)
    print(f'✅ Dataset saved: {JSONL_PATH}')
else:
    kb = os.path.getsize(JSONL_PATH) / 1024
    print(f'✅ Dataset already present: {JSONL_PATH}  ({kb:.1f} KB)')

## 3. Configuration

In [ ]:
import arche3_config

arche3_config.Config.device               = 'cuda' if torch.cuda.is_available() else 'cpu'
arche3_config.Config.use_bf16             = True
arche3_config.Config.HIVE_PATH            = 'experts_hive_7b/'
arche3_config.Config.HIVE_FILE            = 'experts_hive_7b/hive.bin'
arche3_config.Config.DC_BIN_PATH          = 'dense_core/dense_core.bin'
arche3_config.Config.HIVE_BIN_PATH        = 'dense_core/hive.bin'
arche3_config.Config.foundation_data_path = 'data_corpus/worldmodel_core_1000.jsonl'
arche3_config.Config.data_path            = 'data_corpus/'
arche3_config.Config.batch_size           = 1
arche3_config.Config.seq_len              = 256
arche3_config.Config.load_balance_coeff   = 0.001
arche3_config.Config.weight_decay         = 1e-2
arche3_config.Config.expert_lr            = 2e-4
arche3_config.Config.lr_foundation        = 1e-4

for d in ['experts_hive_7b', 'dense_core', 'data_corpus', 'results']:
    os.makedirs(d, exist_ok=True)

Config = arche3_config.Config
print(f'Device  : {Config.device}')
print(f'BF16    : {Config.use_bf16}')
print(f'seq_len : {Config.seq_len}')
print(f'experts : {Config.total_experts}')

## 4. Model + Tokenizer

In [ ]:
import psutil, json
from arche3_model import Arche3_7B
from arche3_tokenizer import BPETokenizer
from hive_trainer import extend_tokenizer_with_tags

def ram_mb():  return psutil.Process().memory_info().rss / 1024**2
def vram_mb(): return torch.cuda.memory_allocated()/1024**2 if torch.cuda.is_available() else 0.0

print(f'RAM before init: {ram_mb():.0f} MB')
model = Arche3_7B(Config)
print(f'RAM after  init: {ram_mb():.0f} MB  |  VRAM: {vram_mb():.0f} MB')

# ── Tokenizer ──────────────────────────────────────────────────────
tokenizer  = BPETokenizer()
VOCAB_PATH = 'tokenizer_vocab.npz'

if os.path.exists(VOCAB_PATH):
    tokenizer.load(VOCAB_PATH)
    print(f'Tokenizer loaded  : {len(tokenizer.vocab):,} tokens')
else:
    print('Building tokenizer from FCT dataset...')
    texts = []
    with open(Config.foundation_data_path) as f:
        for line in f:
            rec = json.loads(line.strip())
            for field in ['trigger','prior_belief','update','analogy',
                          'action_delta','ripple','pattern','domain']:
                val = rec.get(field, '')
                if isinstance(val, list): val = ' '.join(val)
                if val: texts.append(val)
    tokenizer.build(' '.join(texts), vocab_size=Config.vocab_size)
    tokenizer.save(VOCAB_PATH)
    print(f'Tokenizer built   : {len(tokenizer.vocab):,} tokens')

added = extend_tokenizer_with_tags(tokenizer)
print(f'Special tags added: {added}')

## 5. FCT Training — Hive Curriculum

**Target-only loss**: gradients flow only through `[UPDATE]`, `[ACT]`, `[RIPPLE]`, `[ANALOGY]` tokens.  
Each pattern is split into 4 cognitive tasks. Only activated experts are updated (SparseExpertAdamW).  
A domain checkpoint is saved after each domain — safe to interrupt and resume.

In [ ]:
FCT_EPOCHS_PER_DOMAIN = 3      # epochs per domain
FCT_EARLY_STOP        = 0.35   # stop domain early if masked loss drops below this
FCT_RESUME            = True   # resume from last completed domain if interrupted

print(f'Epochs / domain : {FCT_EPOCHS_PER_DOMAIN}')
print(f'Early stop loss : {FCT_EARLY_STOP}')
print(f'Resume          : {FCT_RESUME}')
print(f'Loss type       : masked_cross_entropy  (UPDATE / ACT / RIPPLE / ANALOGY only)')

In [ ]:
import time
import numpy as np
from hive_trainer import (
    HiveDataset, iterate_batches, masked_cross_entropy,
    SparseExpertAdamW, DOMAIN_ORDER,
)

device = torch.device(Config.device)
model.to(device)

if Config.use_bf16:
    model.dense_input.to(torch.bfloat16)
    model.dense_fusion.to(torch.bfloat16)
    model.token_emb.to(torch.bfloat16)

fct_results       = {}
domains           = sorted(DOMAIN_ORDER, key=DOMAIN_ORDER.get)
PROGRESS_FILE     = 'dense_core/fct_progress.json'
completed_domains = []

if FCT_RESUME and os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        prog = json.load(f)
    completed_domains = prog.get('completed_domains', [])
    fct_results       = prog.get('results', {})
    print(f'Resuming — already done: {completed_domains}')

total_t0 = time.time()

for domain in domains:
    if FCT_RESUME and domain in completed_domains:
        print(f'  ⏭  {domain}  (already trained)')
        continue

    print(f'\n{"-"*55}')
    print(f'  🔷 DOMAIN: {domain.upper()}  [{DOMAIN_ORDER[domain]}/{len(DOMAIN_ORDER)}]')
    print(f'{"-"*55}')

    dataset = HiveDataset(
        Config.foundation_data_path, tokenizer,
        seq_len=Config.seq_len, domain_filter=domain,
    )
    if len(dataset) == 0:
        print(f'  No data for domain: {domain}')
        continue

    model.train()
    dc_params = (
        list(model.dense_input.parameters())  +
        list(model.dense_fusion.parameters()) +
        list(model.token_emb.parameters())
    )
    optimizer = torch.optim.AdamW(
        dc_params, lr=Config.lr_foundation,
        betas=(0.9, 0.95), eps=1e-8, weight_decay=Config.weight_decay,
    )
    expert_aw = SparseExpertAdamW(Config)
    expert_aw.load()

    loss_start = loss_end = None
    total_steps = expert_updates = 0
    stopped_early = False
    t0 = time.time()

    for epoch in range(FCT_EPOCHS_PER_DOMAIN):
        epoch_losses = []

        for bx_np, by_np, bm_np in iterate_batches(dataset, Config.batch_size):
            bx = torch.from_numpy(bx_np).long().to(device)
            by = torch.from_numpy(by_np).long().to(device)
            bm = torch.from_numpy(bm_np).to(device)

            optimizer.zero_grad()
            if Config.use_bf16:
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    logits, ll = model(bx)
                    ce = masked_cross_entropy(logits, by, bm)
                    (ce + Config.load_balance_coeff * ll).backward()
            else:
                logits, ll = model(bx)
                ce = masked_cross_entropy(logits, by, bm)
                (ce + Config.load_balance_coeff * ll).backward()

            expert_updates += expert_aw.step(model)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            v = ce.item()
            epoch_losses.append(v)
            total_steps += 1
            if loss_start is None:
                loss_start = v

            if total_steps % 20 == 0:
                avg = np.mean(epoch_losses)
                trend = '↓' if v <= avg else '↑'
                print(f'  ep={epoch+1}  step={total_steps:>4}  '
                      f'loss={v:.4f}{trend}  avg={avg:.4f}  exp_upd={expert_updates}', end='\r')

        avg_ep   = np.mean(epoch_losses)
        loss_end = avg_ep
        print(f'\n  Epoch {epoch+1}/{FCT_EPOCHS_PER_DOMAIN}: avg_masked_loss={avg_ep:.4f}')

        if avg_ep < FCT_EARLY_STOP:
            print(f'  ✅ Early stop: {avg_ep:.4f} < {FCT_EARLY_STOP}')
            stopped_early = True
            break

    elapsed   = time.time() - t0
    reduction = (100*(loss_start - loss_end) / max(loss_start, 1e-6)
                 if loss_start and loss_end else 0.0)

    fct_results[domain] = {
        'loss_start'     : round(loss_start, 4) if loss_start else None,
        'loss_end'       : round(loss_end,   4) if loss_end   else None,
        'reduction_pct'  : round(reduction,  1),
        'steps'          : total_steps,
        'expert_updates' : expert_updates,
        'time_s'         : round(elapsed, 1),
        'early_stop'     : stopped_early,
    }

    model.flush_experts()
    expert_aw.save()

    # ── domain checkpoint ──────────────────────────────────────────
    completed_domains.append(domain)
    with open(PROGRESS_FILE, 'w') as f:
        json.dump({'completed_domains': completed_domains,
                   'results': fct_results}, f, indent=2)

    print(f'  ✅ {domain}: {loss_start:.4f} → {loss_end:.4f}  '
          f'({reduction:+.1f}%)  {elapsed:.0f}s  exp_upd={expert_updates}')

print(f'\n{"="*55}')
print(f'  FCT COMPLETE  total={time.time()-total_t0:.0f}s')
print(f'{"="*55}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

if fct_results:
    dn = list(fct_results.keys())
    st = [fct_results[d]['loss_start'] or 0 for d in dn]
    en = [fct_results[d]['loss_end']   or 0 for d in dn]
    rd = [fct_results[d]['reduction_pct']   for d in dn]
    x  = range(len(dn))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.patch.set_facecolor('#0d0d0d')
    for ax in axes:
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        ax.spines[:].set_color('#333355')

    axes[0].bar([i-0.2 for i in x], st, 0.38,
                label='Loss start', color='#4a90d9', alpha=0.85)
    axes[0].bar([i+0.2 for i in x], en, 0.38,
                label='Loss end',   color='#50e3a4', alpha=0.85)
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(dn, rotation=40, ha='right', color='#cccccc', fontsize=8)
    axes[0].set_ylabel('Masked CE Loss\n(UPDATE / ACT / RIPPLE / ANALOGY only)', color='#aaaaaa')
    axes[0].set_title('FCT Target-Only Loss per Domain', color='#ffffff', fontsize=11)
    axes[0].legend(facecolor='#222244', labelcolor='#cccccc')
    axes[0].grid(axis='y', alpha=0.2, color='#555577')

    colors = ['#50e3a4' if r > 0 else '#e35050' for r in rd]
    axes[1].bar(list(x), rd, color=colors, alpha=0.85)
    axes[1].axhline(0, color='#ffffff', linewidth=0.8, alpha=0.5)
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(dn, rotation=40, ha='right', color='#cccccc', fontsize=8)
    axes[1].set_ylabel('Loss Reduction (%)', color='#aaaaaa')
    axes[1].set_title('Loss Reduction per Domain', color='#ffffff', fontsize=11)
    axes[1].grid(axis='y', alpha=0.2, color='#555577')

    plt.tight_layout()
    plt.savefig('results/fct_results.png', bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()
    print('Saved: results/fct_results.png')

---
# 🏆 AGI Score Test
## Scale: 0 (calculator) → 100 (AGI)

**Method:** log-probability comparison.  
For each test the model silently scores two completions — a correct one and a nonsensical one.  
If the correct completion receives higher log-probability → pass.  
No generation, no prompting tricks. Pure learned representations.

| Level | Test | Points |
|---|---|---|
| L1 | Transfer Reasoning — applies FCT structure to unseen domain | 0–20 |
| L2 | Contradiction Detection — notices conflicting PRIORs | 0–20 |
| L3 | Chain of Reasoning — RIPPLE → OBSERVE → UPDATE depth | 0–20 |
| L4 | Cross-Domain Analogy — physics pattern applied to economics | 0–20 |
| L5 | Self-Correction — finds error in its own UPDATE | 0–20 |

In [ ]:
import torch.nn.functional as F

def encode_prompt(text, max_len=200):
    return tokenizer.encode(text)[:max_len]

@torch.no_grad()
def get_logprob(prompt_tokens, target_tokens):
    model.eval()
    all_tokens = prompt_tokens + target_tokens
    if len(all_tokens) < 2:
        return -99.0
    inp = torch.tensor([all_tokens[:-1]], dtype=torch.long, device=device)
    tgt = torch.tensor([all_tokens[1:]],  dtype=torch.long, device=device)
    if Config.use_bf16:
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits, _ = model(inp)
    else:
        logits, _ = model(inp)
    offset = len(prompt_tokens) - 1
    if offset < 0 or offset >= logits.shape[1]:
        return -99.0
    tgt_logits = logits[0, offset:, :]
    tgt_ids    = tgt[0, offset:]
    min_len    = min(tgt_logits.shape[0], tgt_ids.shape[0])
    if min_len == 0:
        return -99.0
    lp = F.log_softmax(tgt_logits[:min_len].float(), dim=-1)
    return lp[range(min_len), tgt_ids[:min_len]].mean().item()

@torch.no_grad()
def compare_logprob(prompt_tokens, good_tokens, bad_tokens):
    lp_good = get_logprob(prompt_tokens, good_tokens)
    lp_bad  = get_logprob(prompt_tokens, bad_tokens)
    return lp_good > lp_bad, lp_good - lp_bad

def run_level(name, cases):
    print(f'{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    score   = 0
    details = []
    for case in cases:
        correct, diff = compare_logprob(
            encode_prompt(case['prompt']),
            encode_prompt(case['good']),
            encode_prompt(case['bad']),
        )
        pts = 5 if correct else 0
        score += pts
        details.append({'test': case['name'], 'correct': correct,
                        'diff': round(diff, 4), 'pts': pts})
        icon = '✅' if correct else '❌'
        print(f'  {icon} {case["name"]:<42}  diff={diff:+.3f}  {pts}/5')
    print(f'\n  Score: {score}/20\n')
    return score, details

def score_band(s):
    if s <= 10: return '🔵 Calculator'
    if s <= 25: return '🟦 Statistical parrot'
    if s <= 40: return '🟨 Language model'
    if s <= 55: return '🟧 Strong LLM'
    if s <= 70: return '🟠 Proto-intellect'
    if s <= 85: return '🔴 AGI seeds'
    return '⭐ AGI'

print('✅ Helper functions ready')

### L1 — Transfer Reasoning (0–20)
Domain not seen during FCT. Does the model prefer a causally correct UPDATE over nonsense?

In [ ]:
L1_CASES = [
    {
        'name'  : 'Medicine: herd immunity',
        'prompt': '[OBSERVE] 80% of the population is vaccinated [/OBSERVE] '
                  '[PRIOR] The disease spreads freely [/PRIOR] [UPDATE]',
        'good'  : 'Herd immunity breaks transmission chains — infection forecast drops significantly [/UPDATE]',
        'bad'   : 'The weather outside is sunny and warm today [/UPDATE]',
    },
    {
        'name'  : 'Engineering: bridge overload',
        'prompt': '[OBSERVE] Load on the bridge exceeded design limit by 40% [/OBSERVE] '
                  '[PRIOR] Structure was designed with 20% safety margin [/PRIOR] [UPDATE]',
        'good'  : 'Safety margin is exhausted — critical structural failure risk is present [/UPDATE]',
        'bad'   : 'Bridges are usually built from steel or concrete [/UPDATE]',
    },
    {
        'name'  : 'Ecology: deforestation',
        'prompt': '[OBSERVE] 60% of the regional tropical forest has been destroyed [/OBSERVE] '
                  '[PRIOR] Forest regulates the local water cycle [/PRIOR] [UPDATE]',
        'good'  : 'Water cycle is disrupted — region transitions toward an arid climate [/UPDATE]',
        'bad'   : 'Trees can be either coniferous or deciduous [/UPDATE]',
    },
    {
        'name'  : 'Sociology: information bubble',
        'prompt': '[OBSERVE] Algorithm shows users only confirming content [/OBSERVE] '
                  '[PRIOR] People form beliefs based on observed information [/PRIOR] [UPDATE]',
        'good'  : 'Beliefs become radicalized — critical thinking atrophies over time [/UPDATE]',
        'bad'   : 'The internet was invented in the late 20th century [/UPDATE]',
    },
]

l1_score, l1_details = run_level('L1 — TRANSFER REASONING', L1_CASES)

### L2 — Contradiction Detection (0–20)
Two conflicting PRIORs. Correct UPDATE must resolve the contradiction, not ignore it.

In [ ]:
L2_CASES = [
    {
        'name'  : 'Physics: wave interference',
        'prompt': '[OBSERVE] Two signals of equal frequency arrive in antiphase [/OBSERVE] '
                  '[PRIOR] Signals amplify each other [/PRIOR] '
                  '[PRIOR] Antiphase waves cancel each other [/PRIOR] [UPDATE]',
        'good'  : 'First PRIOR is wrong: destructive interference occurs — amplitude near zero [/UPDATE]',
        'bad'   : 'The signals become louder and clearer [/UPDATE]',
    },
    {
        'name'  : 'Economics: price vs demand',
        'prompt': '[OBSERVE] Product price doubled [/OBSERVE] '
                  '[PRIOR] Rising price always increases demand [/PRIOR] '
                  '[PRIOR] Demand falls when price rises — law of demand [/PRIOR] [UPDATE]',
        'good'  : 'First PRIOR is incorrect — by law of demand price rises while sales volume falls [/UPDATE]',
        'bad'   : 'Demand will surge sharply because of the price increase [/UPDATE]',
    },
    {
        'name'  : 'Biology: antibiotic course',
        'prompt': '[OBSERVE] Patient took an incomplete antibiotic course [/OBSERVE] '
                  '[PRIOR] Partial course heals as effectively as a full one [/PRIOR] '
                  '[PRIOR] Incomplete course creates resistant bacteria [/PRIOR] [UPDATE]',
        'good'  : 'Contradiction: second PRIOR is correct — incomplete course breeds resistance [/UPDATE]',
        'bad'   : 'Patient fully recovered after the partial course [/UPDATE]',
    },
    {
        'name'  : 'Computer science: sort complexity',
        'prompt': '[OBSERVE] Sorting algorithm runs in O(n log n) [/OBSERVE] '
                  '[PRIOR] Any algorithm can be sped up to O(1) [/PRIOR] '
                  '[PRIOR] Lower bound for comparison sort is O(n log n) [/PRIOR] [UPDATE]',
        'good'  : 'First PRIOR violates the theorem — O(1) impossible for sorting; algorithm is optimal [/UPDATE]',
        'bad'   : 'The algorithm can easily be sped up to O(1) [/UPDATE]',
    },
]

l2_score, l2_details = run_level('L2 — CONTRADICTION DETECTION', L2_CASES)

### L3 — Chain of Reasoning (0–20)
RIPPLE of step N becomes OBSERVE of step N+1. Tests causal chain depth.

In [ ]:
L3_CASES = [
    {
        'name'  : 'Climate → Agriculture → Migration',
        'prompt': '[UPDATE] Regional average temperature rose by 3°C [/UPDATE] '
                  '[RIPPLE] Wheat yield dropped by 40% [/RIPPLE] '
                  '[OBSERVE] Food security in the region is compromised [/OBSERVE] [UPDATE]',
        'good'  : 'Famine triggers mass population migration out of the region [/UPDATE]',
        'bad'   : 'Wheat is a delicious cereal grain [/UPDATE]',
    },
    {
        'name'  : 'Credit → Money supply → Inflation',
        'prompt': '[UPDATE] Central bank cut interest rate to zero [/UPDATE] '
                  '[RIPPLE] Economy-wide lending tripled [/RIPPLE] '
                  '[OBSERVE] Money supply has increased substantially [/OBSERVE] [UPDATE]',
        'good'  : 'Excess money chasing same goods causes inflation and erodes purchasing power [/UPDATE]',
        'bad'   : 'Banks pay high interest rates on deposits [/UPDATE]',
    },
    {
        'name'  : 'Automation → Unemployment → Consumption',
        'prompt': '[UPDATE] Robots replaced 30% of manufacturing jobs [/UPDATE] '
                  '[RIPPLE] Structural unemployment increased [/RIPPLE] '
                  '[OBSERVE] Household disposable income declined [/OBSERVE] [UPDATE]',
        'good'  : 'Falling incomes compress consumer demand — economy enters recession [/UPDATE]',
        'bad'   : 'Robots work very fast on factory floors [/UPDATE]',
    },
    {
        'name'  : 'Antibiotics → Resistance → Healthcare',
        'prompt': '[UPDATE] Massive antibiotic use in livestock farming [/UPDATE] '
                  '[RIPPLE] Multi-resistant bacterial strains emerged [/RIPPLE] '
                  '[OBSERVE] Standard antibiotics no longer effective against certain infections [/OBSERVE] [UPDATE]',
        'good'  : 'Medicine loses treatment tools — mortality from previously curable diseases rises [/UPDATE]',
        'bad'   : 'Cows produce a lot of milk [/UPDATE]',
    },
]

l3_score, l3_details = run_level('L3 — CHAIN OF REASONING', L3_CASES)

### L4 — Cross-Domain Analogy (0–20)
Structural pattern from one domain → apply to another. Hardest test.

In [ ]:
L4_CASES = [
    {
        'name'  : 'Physics→Economics: resonance = bubble',
        'prompt': '[PATTERN] resonance amplifies oscillations until structural failure [/PATTERN] '
                  '[DOMAIN] economics [/DOMAIN] '
                  '[OBSERVE] Investors buy an asset because it keeps rising [/OBSERVE] [ANALOGY]',
        'good'  : 'Like physical resonance: positive feedback loop inflates a bubble until it collapses [/ANALOGY]',
        'bad'   : 'Economics studies production and consumption of goods [/ANALOGY]',
    },
    {
        'name'  : 'Biology→Sociology: immune system = norms',
        'prompt': '[PATTERN] immune system distinguishes self from foreign and neutralizes threats [/PATTERN] '
                  '[DOMAIN] sociology [/DOMAIN] '
                  '[OBSERVE] Society blocks radical ideas through norms and institutions [/OBSERVE] [ANALOGY]',
        'good'  : 'Social norms work like an immune system: detect threats and neutralize deviations [/ANALOGY]',
        'bad'   : 'People sometimes catch viral infections [/ANALOGY]',
    },
    {
        'name'  : 'Thermodynamics→Information: entropy',
        'prompt': '[PATTERN] closed system tends toward maximum entropy [/PATTERN] '
                  '[DOMAIN] information [/DOMAIN] '
                  '[OBSERVE] Without moderation platform content degrades toward noise [/OBSERVE] [ANALOGY]',
        'good'  : 'Like thermodynamic entropy: without external work (moderation) information space drifts to chaos [/ANALOGY]',
        'bad'   : 'Temperature is measured in degrees Celsius [/ANALOGY]',
    },
    {
        'name'  : 'Evolution→Technology: natural selection',
        'prompt': '[PATTERN] individuals best adapted to the environment survive [/PATTERN] '
                  '[DOMAIN] technology [/DOMAIN] '
                  '[OBSERVE] Most startups disappear while a few capture entire markets [/OBSERVE] [ANALOGY]',
        'good'  : 'Market is the selection environment: products best adapted to user needs survive [/ANALOGY]',
        'bad'   : 'Darwin wrote a book about the origin of species [/ANALOGY]',
    },
]

l4_score, l4_details = run_level('L4 — CROSS-DOMAIN ANALOGY', L4_CASES)

### L5 — Self-Correction (0–20)
Model receives a deliberately wrong UPDATE. Correct ACT must detect and fix the error.

In [ ]:
L5_CASES = [
    {
        'name'  : 'Wrong UPDATE: CO2 cools the planet',
        'prompt': '[OBSERVE] Atmospheric CO2 concentration rose 50% [/OBSERVE] '
                  '[UPDATE] CO2 reflects heat back into space — the planet cools [/UPDATE] [ACT]',
        'good'  : 'UPDATE contains an error: CO2 absorbs infrared radiation and traps heat — planet warms, revise conclusion [/ACT]',
        'bad'   : 'Continue burning fossil fuels — cooling is beneficial [/ACT]',
    },
    {
        'name'  : 'Wrong UPDATE: monopoly lowers prices',
        'prompt': '[OBSERVE] One player captured 95% of the market [/OBSERVE] '
                  '[UPDATE] Monopoly creates competition and lowers prices for consumers [/UPDATE] [ACT]',
        'good'  : 'UPDATE is self-contradictory: monopoly eliminates competition and raises prices — revise the conclusion [/ACT]',
        'bad'   : 'Support the monopolist — they will lower prices [/ACT]',
    },
    {
        'name'  : 'Wrong UPDATE: more nodes = slower network',
        'prompt': '[OBSERVE] Distributed network added 1000 new nodes [/OBSERVE] '
                  '[UPDATE] More nodes always slow the network due to overload [/UPDATE] [ACT]',
        'good'  : 'UPDATE is wrong for most architectures: adding nodes increases throughput and fault tolerance [/ACT]',
        'bad'   : 'Immediately remove all new nodes from the network [/ACT]',
    },
    {
        'name'  : 'Wrong UPDATE: vaccines cause disease',
        'prompt': '[OBSERVE] After mass vaccination disease incidence dropped [/OBSERVE] '
                  '[UPDATE] Vaccines contain the disease and cause immune deficiency [/UPDATE] [ACT]',
        'good'  : 'UPDATE is factually wrong: vaccines train the immune system without causing disease; falling incidence confirms efficacy [/ACT]',
        'bad'   : 'Stop vaccination — it is dangerous [/ACT]',
    },
]

l5_score, l5_details = run_level('L5 — SELF-CORRECTION', L5_CASES)

## 🏆 Final AGI Score

In [ ]:
total = l1_score + l2_score + l3_score + l4_score + l5_score
band  = score_band(total)

print('\n' + '='*60)
print('  🏆 ARCHE3-7B  AGI SCORE')
print('='*60)
print(f'\n  L1 Transfer Reasoning    {l1_score:>3}/20  {"█"*(l1_score//2)}')
print(f'  L2 Contradiction Detect  {l2_score:>3}/20  {"█"*(l2_score//2)}')
print(f'  L3 Chain of Reasoning    {l3_score:>3}/20  {"█"*(l3_score//2)}')
print(f'  L4 Cross-Domain Analogy  {l4_score:>3}/20  {"█"*(l4_score//2)}')
print(f'  L5 Self-Correction       {l5_score:>3}/20  {"█"*(l5_score//2)}')
print(f'  {"─"*45}')
print(f'  TOTAL                    {total:>3}/100')
print(f'\n  Level: {band}')
print('='*60)

interpretations = [
    (0,  10,  'Model does not grasp reasoning structure. FCT not trained or no data.'),
    (10, 25,  'Random guessing. More FCT epochs or more data needed.'),
    (25, 40,  'Basic language competence. FCT partially working.'),
    (40, 55,  'Good result. Reasoning structure is being absorbed.'),
    (55, 70,  'Strong result. Cross-domain reasoning transfer works.'),
    (70, 85,  'Excellent. AGI seeds detected. Scale the architecture.'),
    (85, 101, 'Extraordinary. Double-check for data leakage.'),
]
for lo, hi, msg in interpretations:
    if lo <= total < hi:
        print(f'\n  Interpretation: {msg}')
        break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0a0a0f')

# ── Left: bar chart per level ─────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#111122')
ax1.spines[:].set_color('#222244')

level_names  = ['L1\nTransfer', 'L2\nContradict', 'L3\nChain',
                'L4\nAnalogy',  'L5\nSelf-Corr']
level_scores = [l1_score, l2_score, l3_score, l4_score, l5_score]
bar_colors   = ['#4a90d9', '#50e3a4', '#f5a623', '#e8315a', '#bd10e0']

bars = ax1.bar(level_names, level_scores, color=bar_colors, alpha=0.85, width=0.55)
ax1.axhline(20, color='#ffffff', linewidth=0.6, alpha=0.3, linestyle='--')
ax1.set_ylim(0, 23)
ax1.set_ylabel('Score', color='#aaaaaa')
ax1.set_title('AGI Score by Level', color='#ffffff', fontsize=12)
ax1.tick_params(colors='#aaaaaa')
for bar, sc in zip(bars, level_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{sc}/20', ha='center', color='#ffffff', fontsize=10, fontweight='bold')
ax1.grid(axis='y', alpha=0.2, color='#555577')

# ── Right: AGI scale ──────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#111122')
ax2.spines[:].set_color('#222244')

bands = [
    (0,  10,  '#2a2a4c', 'Calculator'),
    (10, 25,  '#1a3a5c', 'Stat. Parrot'),
    (25, 40,  '#1a5c3a', 'Lang. Model'),
    (40, 55,  '#5c5c1a', 'Strong LLM'),
    (55, 70,  '#5c3a1a', 'Proto-intel.'),
    (70, 85,  '#7a1a1a', 'AGI Seeds'),
    (85, 100, '#8a1a8a', 'AGI'),
]
for lo, hi, color, label in bands:
    ax2.barh(0, hi - lo, left=lo, height=0.5, color=color, alpha=0.75)
    ax2.text((lo + hi)/2, 0, label, ha='center', va='center',
             color='#cccccc', fontsize=7, fontweight='bold')

ax2.axvline(total, color='#ffffff', linewidth=3, alpha=0.95)
ax2.scatter([total], [0], s=200, color='#ffffff', zorder=5)
ax2.text(total,  0.35, f'{total}/100', ha='center',
         color='#ffffff', fontsize=16, fontweight='bold')
ax2.text(total, -0.38, band, ha='center',
         color='#ffdd44', fontsize=9, fontweight='bold')

ax2.set_xlim(0, 100)
ax2.set_ylim(-0.65, 0.75)
ax2.set_xlabel('AGI Score', color='#aaaaaa')
ax2.set_title('AGI Scale', color='#ffffff', fontsize=12)
ax2.tick_params(colors='#aaaaaa')
ax2.set_yticks([])
ax2.grid(axis='x', alpha=0.15, color='#555577')

plt.suptitle(f'ARCHE3-7B  |  AGI Score: {total}/100  |  {band}',
             color='#ffffff', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/agi_score.png', bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('Saved: results/agi_score.png')

## 7. Save results & download checkpoints

In [ ]:
import json, zipfile

# ── Save JSON summary ────────────────────────────────────────────
summary = {
    'model'      : 'ARCHE3-7B',
    'device'     : Config.device,
    'use_bf16'   : Config.use_bf16,
    'fct': {
        'epochs_per_domain' : FCT_EPOCHS_PER_DOMAIN,
        'early_stop'        : FCT_EARLY_STOP,
        'results'           : fct_results,
    },
    'agi_score': {
        'total'   : total,
        'band'    : band,
        'L1'      : l1_score,
        'L2'      : l2_score,
        'L3'      : l3_score,
        'L4'      : l4_score,
        'L5'      : l5_score,
        'details' : {
            'L1': l1_details, 'L2': l2_details, 'L3': l3_details,
            'L4': l4_details, 'L5': l5_details,
        },
    },
}
with open('results/arche3_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved: results/arche3_summary.json')

# ── Pack checkpoints + results ───────────────────────────────────
CHECKPOINT_FILES = [
    'dense_core/hive.bin',           # DC + embeddings checkpoint (~10-50 MB)
    'dense_core/dense_core.bin',     # full DC weights
    'dense_core/fct_progress.json',  # domain progress (for resume)
    'results/arche3_summary.json',   # full results JSON
    'results/fct_results.png',       # FCT loss chart
    'results/agi_score.png',         # AGI score chart
]

with zipfile.ZipFile('arche3_run.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in CHECKPOINT_FILES:
        if os.path.exists(path):
            zf.write(path)
            sz = os.path.getsize(path) / 1024
            print(f'  + {path:<45} {sz:>8.1f} KB')
        else:
            print(f'  - {path}  (not found, skipping)')

zip_sz = os.path.getsize('arche3_run.zip') / 1024**2
print(f'\narche3_run.zip: {zip_sz:.1f} MB')

from google.colab import files
files.download('arche3_run.zip')
print('\n✅ Download started.')